# La H200 de la USFQ — LLM y embeddings desde Python

**MMIA 6013 · Semana 2 · preliminar de la sesión 07 (Bases de datos vectoriales)**

La universidad tiene una **NVIDIA H200** sirviendo modelos detrás de APIs compatibles con
la de OpenAI. Para este curso significa tres cosas:

- **Los embeddings del Taller 2 no cuestan nada.** Un corpus de miles de fragmentos se
  paga en dinero contra una API comercial; aquí se paga en segundos de una GPU encendida.
- **Los datos no salen de la universidad.** Es la diferencia que importa cuando el corpus
  del proyecto final son documentos internos.
- **Hay que estar en la red de la USFQ.** Con GlobalProtect desconectada no hay endpoint,
  y ese es el primer error que van a encontrar.

Este cuaderno trae **una sola clase**, `H200`, y la usa para las dos cosas que la semana 2
necesita: **calcular embeddings** y **llamar al LLM**.

Requisitos: `numpy`. Nada más — la clase usa solo la biblioteca estándar, así que no hay
que instalar `openai` ni `requests` en treinta portátiles.

## Qué hay servido, y por qué son dos servidores distintos

Verificado el **2026-09-22** contra `172.28.230.10`:

| Puerto  | Servidor | Qué sirve                                   | Para qué sirve aquí |
|---------|----------|---------------------------------------------|---------------------|
| `12555` | vLLM     | `zai-org/GLM-5.3-Flash` (ventana de 1 048 576 tokens) | el LLM |
| `12559` | vLLM     | el mismo modelo, segunda réplica             | el LLM |
| `11434` | Ollama   | 14 modelos, entre ellos `bge-m3` y `nomic-embed-text` | los embeddings |

**El endpoint de vLLM no hace embeddings**: `/v1/embeddings` en el 12555 responde `404`.
Los dos modelos de embeddings viven en el Ollama del 11434, así que un cliente que solo
mire el puerto del LLM no los encuentra nunca. Por eso la clase **barre varios puertos**.

De los dos modelos de embeddings, el que usa este cuaderno por defecto es **`bge-m3`**:
1024 dimensiones y **multilingüe**, que es lo que hace falta con un corpus en español.
`nomic-embed-text` son 768 dimensiones y está entrenado sobre todo en inglés.

## 🧑‍🏫 Notas docentes

- **Este archivo no lo procesa `build_notebooks.py`**: su patrón es `sX-DIA.py` y éste no
  lo cumple, a propósito. Es material de apoyo del profesor, no la actividad del día.
  Se regenera con
  `.venv/bin/jupytext --to ipynb --output curso/actividades/semana-2/h200-usfq-profesor.ipynb curso/actividades/_fuentes/h200-usfq.py`
- **Ningún id de modelo está escrito a mano.** vLLM publica con sufijo de versión y el
  laboratorio cambia lo que sirve: el 2026-09-22 el 12555 dejó de servir
  `deepseek-ai/DeepSeek-V4-Flash` —lo que dice `~/.claude/commands/h200.md`— y sirve
  `zai-org/GLM-5.3-Flash`. La clase pregunta al endpoint y decide.
- **El descubrimiento NO prueba los modelos generativos, y es deliberado.** El 11434 es un
  Ollama que **carga el modelo en la GPU cuando le llega la primera petición**: tiene
  `llama3.1:405b` y `qwen3:235b` en el catálogo sin cargar. Mandarle una petición de
  prueba a cada modelo para ver cuál es de embeddings levantaría un modelo de 405 000
  millones de parámetros para averiguar algo que su nombre ya decía. Se prueban solo los
  candidatos por nombre.
- **El interruptor de razonamiento hace lo contrario de lo que parece**, y está medido
  abajo en la Parte 2. GLM razona siempre; `enable_thinking: False` no lo apaga, solo
  impide que vLLM separe la cadena de razonamiento, y entonces **la respuesta sale con el
  monólogo interno pegado delante, en inglés**. El default de la clase es dejarla
  separada.
- **Si el aula corre esto a la vez**, la GPU es una sola. Los embeddings son baratos
  (39 ms por texto con `bge-m3` ya cargado); el LLM no: una respuesta de dos frases
  consume ~500 tokens de salida porque el razonamiento se paga del mismo presupuesto.

## La clase

Cuatro decisiones que conviene leer antes del código, porque son las que la hacen
sobrevivir al semestre:

1. **Descubre, no supone.** Recorre los puertos, pregunta `/v1/models` y decide qué es
   cada modelo. Cuando el laboratorio cambie el modelo —ya pasó— esto sigue funcionando.
2. **Descubre sin despertar a nadie.** Solo prueba los candidatos a embeddings; a un
   modelo generativo no le manda nada durante el descubrimiento.
3. **Falla diciendo qué hacer.** Un `timeout` contra una IP interna casi siempre es la
   VPN, no el servidor, así que eso es lo que dice el mensaje.
4. **Devuelve medidas, no solo texto.** `chat()` devuelve tokens, latencia y razón de
   parada, porque el resto del curso se corrige midiendo.

In [1]:
import json
import time
import urllib.error
import urllib.request

import numpy as np


class ErrorH200(RuntimeError):
    """Error del cliente. El mensaje dice qué hacer, no solo qué falló."""


class H200:
    """Cliente de los endpoints de la H200 de la USFQ. Solo biblioteca estándar.

    Uso mínimo:

        h = H200()
        h.diagnostico()                     # qué hay servido y dónde
        h.preguntar("¿Qué es un índice HNSW?")
        h.embeddings(["hola", "adiós"])     # (2, 1024) float32, normalizados
    """

    HOST = "172.28.230.10"
    # Dos servidores distintos: vLLM sirve el LLM (un modelo por proceso, de ahí las dos
    # réplicas) y Ollama sirve, entre otros, los dos modelos de embeddings.
    PUERTOS = (12555, 12559, 11434)
    # Un nombre no es una garantía, pero sí es una pista suficiente para decidir A QUIÉN se
    # le manda la petición de prueba.
    PISTAS_EMBEDDINGS = ("embed", "bge", "e5-", "gte", "minilm", "nomic", "mxbai")
    # Con varios candidatos, este orden decide. bge-m3 primero porque es multilingüe y el
    # corpus del curso está en español. Si no hay ninguno de la lista, se usa el que haya.
    PREFERIDOS_EMBEDDINGS = ("bge-m3", "multilingual-e5", "mxbai", "nomic")

    def __init__(self, host=HOST, puertos=PUERTOS, api_key="local", timeout=180,
                 timeout_descubrimiento=6):
        self.host = host
        self.puertos = tuple(puertos)
        self.api_key = api_key          # local: no se valida. NUNCA pongas una clave real.
        self.timeout = timeout
        self.timeout_descubrimiento = timeout_descubrimiento
        self._catalogo = None           # {(puerto, id): "chat" | "embeddings"}
        self._fijado = {}               # rol -> (puerto, modelo) elegido a mano
        self._dimension = None

    # ── transporte ────────────────────────────────────────────────────────────────────
    def _pedir(self, puerto, ruta, cuerpo=None, timeout=None):
        """GET si `cuerpo` es None, POST si no. Devuelve (codigo, json)."""
        datos = None if cuerpo is None else json.dumps(cuerpo).encode()
        pet = urllib.request.Request(
            f"http://{self.host}:{puerto}/{ruta.lstrip('/')}", data=datos,
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {self.api_key}"})
        try:
            with urllib.request.urlopen(pet, timeout=timeout or self.timeout) as r:
                return r.status, json.loads(r.read())
        except urllib.error.HTTPError as err:
            # Un 4xx es una RESPUESTA, no una avería: en el descubrimiento es justo la
            # señal que separa un modelo de embeddings de uno generativo.
            try:
                return err.code, json.loads(err.read())
            except Exception:
                return err.code, {"error": {"message": f"HTTP {err.code}"}}
        except urllib.error.URLError as err:
            raise ErrorH200(
                f"No hay respuesta de {self.host}:{puerto} ({err.reason}).\n"
                f"Contra una IP interna esto casi siempre es la VPN, no el servidor:\n"
                f"  1. Conecta GlobalProtect y espera a que diga «Connected».\n"
                f"  2. Comprueba la ruta:  route -n get {self.host}\n"
                f"     Si la interfaz que sale NO es un `utun`, el túnel no está enrutando.\n"
                f"  3. Prueba el puerto:   nc -z -w 5 {self.host} {puerto}"
            ) from err

    # ── descubrimiento ────────────────────────────────────────────────────────────────
    def catalogo(self, refrescar=False):
        """{(puerto, modelo): rol} con rol en {'chat', 'embeddings'}.

        A los candidatos a embeddings se les CONFIRMA el rol con una petición de una
        palabra; a los generativos no se les manda nada. El motivo es que el 11434 es un
        Ollama y **carga el modelo en la GPU al recibir la primera petición**: probar su
        catálogo entero levantaría `llama3.1:405b` para averiguar lo que su nombre ya dice.
        Cuando un puerto sirve uno o dos modelos —el caso de vLLM— sí se prueba, porque
        ahí el modelo ya está residente y la prueba es gratis.
        """
        if self._catalogo is not None and not refrescar:
            return self._catalogo

        encontrado, mudos = {}, []
        for puerto in self.puertos:
            try:
                codigo, cuerpo = self._pedir(puerto, "v1/models",
                                             timeout=self.timeout_descubrimiento)
            except ErrorH200:
                mudos.append(puerto)
                continue
            if codigo != 200:
                continue
            servidos = [e["id"] for e in cuerpo.get("data", [])]
            for modelo in servidos:
                parece = any(p in modelo.lower() for p in self.PISTAS_EMBEDDINGS)
                if parece or len(servidos) <= 2:
                    codigo_emb, _ = self._pedir(puerto, "v1/embeddings",
                                                {"model": modelo, "input": "ping"},
                                                timeout=self.timeout_descubrimiento)
                    rol = "embeddings" if codigo_emb == 200 else "chat"
                else:
                    rol = "chat"        # generativo por descarte: no se le despierta
                encontrado[(puerto, modelo)] = rol

        if not encontrado:
            raise ErrorH200(
                f"Ningún puerto de {self.host} respondió "
                f"({', '.join(map(str, mudos)) or 'ninguno probado'}).\n"
                f"Conecta GlobalProtect; si ya está conectada, avisa al laboratorio.")
        self._catalogo = encontrado
        return encontrado

    def _resolver(self, rol):
        if rol in self._fijado:
            return self._fijado[rol]
        catalogo = self.catalogo()
        candidatos = [(p, m) for (p, m), r in catalogo.items() if r == rol]
        if not candidatos:
            hay = ", ".join(f"{m} ({r}, :{p})" for (p, m), r in sorted(catalogo.items()))
            raise ErrorH200(f"El endpoint no sirve ningún modelo de tipo '{rol}'. Hay: {hay}")

        if rol == "embeddings":
            for preferido in self.PREFERIDOS_EMBEDDINGS:
                iguales = [pm for pm in candidatos if preferido in pm[1].lower()]
                if iguales:
                    return sorted(iguales)[0]
            return sorted(candidatos)[0]

        # Para el LLM se prefiere un endpoint DEDICADO —el que sirve un solo modelo— antes
        # que un catálogo como el de Ollama: en un catálogo, pedir un modelo frío significa
        # cargarlo en la GPU, y eso con el aula conectada se nota.
        por_puerto = {}
        for (p, m) in catalogo:
            por_puerto[p] = por_puerto.get(p, 0) + 1
        return sorted(candidatos, key=lambda pm: (por_puerto[pm[0]], pm[1]))[0]

    def usar(self, rol, modelo, puerto=None):
        """Fija a mano el modelo de un rol: `h.usar('chat', 'qwen3:32b', 11434)`."""
        for (p, m) in self.catalogo():
            if m == modelo and (puerto is None or p == puerto):
                self._fijado[rol] = (p, m)
                self._dimension = None if rol == "embeddings" else self._dimension
                return (p, m)
        raise ErrorH200(f"'{modelo}' no está servido. Mira h.diagnostico().")

    @property
    def modelo_chat(self):
        return self._resolver("chat")[1]

    @property
    def modelo_embeddings(self):
        return self._resolver("embeddings")[1]

    def cargados(self, puerto=11434):
        """Modelos que Ollama tiene EN LA GPU ahora mismo. [] si el puerto no es Ollama.

        Importa antes de una clase: pedirle a Ollama un modelo que no está cargado lo carga,
        y con 30 estudiantes esperando eso es un minuto largo de silencio.
        """
        try:
            codigo, cuerpo = self._pedir(puerto, "api/ps", timeout=self.timeout_descubrimiento)
        except ErrorH200:
            return []
        if codigo != 200:
            return []
        return [(m["name"], m.get("size_vram", 0) / 1e9) for m in cuerpo.get("models", [])]

    def diagnostico(self):
        """Imprime qué hay servido y dónde. Es la primera celda que hay que correr."""
        try:
            catalogo = self.catalogo(refrescar=True)
        except ErrorH200 as err:
            print(f"✗ sin endpoint\n{err}")
            return False
        print(f"✓ {self.host} · {len(catalogo)} modelo(s) en "
              f"{len({p for p, _ in catalogo})} puerto(s)")
        calientes = dict(self.cargados())
        for (puerto, modelo), rol in sorted(catalogo.items()):
            marca = f"  ← en GPU, {calientes[modelo]:.1f} GB" if modelo in calientes else ""
            print(f"   :{puerto}  {rol:11s}  {modelo}{marca}")
        print(f"\n   chat       → {self.modelo_chat}")
        print(f"   embeddings → {self.modelo_embeddings}")
        return True

    # ── el LLM ────────────────────────────────────────────────────────────────────────
    def chat(self, prompt, system=None, temperature=0.3, max_tokens=2048,
             separar_razonamiento=True):
        """Una llamada al LLM. Devuelve texto, razonamiento, tokens y latencia.

        Sobre `separar_razonamiento` —medido el 2026-09-22 sobre GLM-5.3-Flash—: **no es un
        interruptor de razonamiento**. El modelo razona siempre. Lo que decide es si vLLM
        devuelve la cadena de razonamiento en su propio campo (`True`, y entonces `content`
        trae solo la respuesta) o la deja dentro de `content` (`False`, y entonces la
        respuesta viene con el monólogo interno pegado delante, normalmente en inglés).
        El default es el que sirve para algo.

        `max_tokens` no es generoso por gusto: el razonamiento se paga del MISMO
        presupuesto, y una respuesta de dos frases consumió 514 tokens en la medición.
        Con 400 se corta a media palabra y `razon_de_parada` sale como `length`.
        """
        puerto, modelo = self._resolver("chat")
        mensajes = ([{"role": "system", "content": system}] if system else []) + \
                   [{"role": "user", "content": prompt}]
        cuerpo = {"model": modelo, "messages": mensajes, "temperature": temperature,
                  "max_tokens": max_tokens,
                  # vLLM solo lo entiende aquí dentro. (Con el SDK de OpenAI habría que
                  # pasarlo por `extra_body`; con HTTP crudo va en el cuerpo y ya.)
                  "chat_template_kwargs": {"enable_thinking": bool(separar_razonamiento)}}

        t0 = time.perf_counter()
        codigo, respuesta = self._pedir(puerto, "v1/chat/completions", cuerpo)
        latencia = time.perf_counter() - t0
        if codigo != 200:
            raise ErrorH200(f"{codigo} · {respuesta.get('error', {}).get('message', respuesta)}")

        msg = respuesta["choices"][0]["message"]
        texto = (msg.get("content") or "").strip()
        # Unos servidores lo llaman `reasoning` y otros `reasoning_content`; y algún modelo
        # deja `content` vacío y pone todo en el campo de razonamiento, así que si no hay
        # respuesta se usa eso antes que devolver una cadena vacía.
        razonamiento = (msg.get("reasoning") or msg.get("reasoning_content") or "").strip()
        if not texto and razonamiento:
            texto = razonamiento
        uso = respuesta.get("usage", {})
        return {"texto": texto, "razonamiento": razonamiento, "modelo": modelo,
                "tokens_entrada": uso.get("prompt_tokens"),
                "tokens_salida": uso.get("completion_tokens"),
                "razon_de_parada": respuesta["choices"][0].get("finish_reason"),
                "latencia_s": latencia}

    def preguntar(self, prompt, **kw):
        """Atajo: solo el texto. Para cuando no hace falta medir."""
        return self.chat(prompt, **kw)["texto"]

    # ── los embeddings ────────────────────────────────────────────────────────────────
    def embeddings(self, textos, lote=32, normalizar=True, mostrar=False):
        """Embeddings de una lista de textos → array (n, d) float32.

        `normalizar=True` deja cada vector con norma 1, y eso NO es cosmética: con vectores
        normalizados el producto punto ES el coseno, así que una búsqueda semántica se
        escribe como una multiplicación de matrices y los índices de la sesión 07 —HNSW,
        IVF— comparan lo mismo que tú. (`bge-m3` ya los devuelve normalizados; otros
        modelos no, y el código no debería depender de cuál está servido hoy.)

        Se manda por lotes porque el servidor tiene un techo por petición y porque un fallo
        a mitad de un corpus de 5 000 fragmentos debe costar un lote, no el corpus.
        """
        if isinstance(textos, str):
            textos = [textos]
        textos = list(textos)
        if not textos:
            return np.zeros((0, self.dimension()), dtype=np.float32)

        puerto, modelo = self._resolver("embeddings")
        vectores = []
        for inicio in range(0, len(textos), lote):
            trozo = textos[inicio:inicio + lote]
            codigo, respuesta = self._pedir(puerto, "v1/embeddings",
                                            {"model": modelo, "input": trozo})
            if codigo != 200:
                raise ErrorH200(
                    f"{codigo} en el lote {inicio}–{inicio + len(trozo)} · "
                    f"{respuesta.get('error', {}).get('message', respuesta)}")
            # El orden NO se da por supuesto: la respuesta trae `index` y se usa.
            for fila in sorted(respuesta["data"], key=lambda d: d["index"]):
                vectores.append(fila["embedding"])
            if mostrar:
                print(f"   {min(inicio + lote, len(textos)):>5d}/{len(textos)} textos")

        X = np.asarray(vectores, dtype=np.float32)
        if normalizar:
            X /= np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-12)
        return X

    def dimension(self):
        """Dimensión del espacio de embeddings. Se pregunta una vez y se recuerda."""
        if self._dimension is None:
            self._dimension = int(self.embeddings(["dimensión"], normalizar=False).shape[1])
        return self._dimension


print("clase H200 lista ·", ", ".join(m for m in dir(H200) if not m.startswith("_")))

clase H200 lista · HOST, PISTAS_EMBEDDINGS, PREFERIDOS_EMBEDDINGS, PUERTOS, cargados, catalogo, chat, diagnostico, dimension, embeddings, modelo_chat, modelo_embeddings, preguntar, usar


## 1 · Qué hay servido hoy

Primera celda de toda clase. Si esto falla, **no es el código**: es la VPN.

La marca «en GPU» sale de `/api/ps` de Ollama y dice qué modelos están calientes. Un
modelo frío se carga al pedirlo, y eso son decenas de segundos con el aula esperando.

In [2]:
h = H200()
HAY_H200 = h.diagnostico()

✓ 172.28.230.10 · 16 modelo(s) en 3 puerto(s)
   :11434  embeddings   bge-m3:latest  ← en GPU, 1.3 GB
   :11434  chat         gemma3:27b
   :11434  chat         gpt-oss:120b
   :11434  chat         granite3.3:latest
   :11434  chat         llama3.1:405b
   :11434  chat         llama3.2-vision:latest
   :11434  chat         llama4:17b-scout-16e-instruct-q4_K_M
   :11434  chat         llama4:maverick
   :11434  embeddings   nomic-embed-text:latest  ← en GPU, 0.6 GB
   :11434  chat         qwen3:235b
   :11434  chat         qwen3:235b-a22b
   :11434  chat         qwen3:32b
   :11434  chat         qwen3:8b
   :11434  chat         qwen3:latest
   :12555  chat         zai-org/GLM-5.3-Flash
   :12559  chat         zai-org/GLM-5.3-Flash

   chat       → zai-org/GLM-5.3-Flash
   embeddings → bge-m3:latest


## 2 · El LLM

La misma API de la semana 1 —mensajes, `temperature`, `max_tokens`— contra una GPU de la
universidad en vez de contra una API que cobra por token. Lo que cambia es el precio y
quién ve los datos, no el modelo mental.

In [3]:
if HAY_H200:
    r = h.chat("En dos frases: ¿qué es una base de datos vectorial y para qué sirve?",
               system="Eres profesor de un curso de maestría en IA generativa. Responde "
                      "en español, breve y sin listas.")
    print(f"modelo   : {r['modelo']}")
    print(f"medidas  : {r['tokens_entrada']} tok entrada · {r['tokens_salida']} salida · "
          f"{r['latencia_s']:.2f} s · parada por {r['razon_de_parada']}")
    print(f"razonó   : {len(r['razonamiento'])} caracteres, aparte de la respuesta")
    print(f"respuesta:\n{r['texto']}")

modelo   : zai-org/GLM-5.3-Flash
medidas  : 58 tok entrada · 281 salida · 2.21 s · parada por stop
razonó   : 749 caracteres, aparte de la respuesta
respuesta:
Una base de datos vectorial es un sistema que almacena datos (texto, imágenes, audio) representados como vectores numéricos de alta dimensión —llamados embeddings— que capturan su significado semántico, y que permite buscar por similitud en lugar de por coincidencias exactas. Su principal uso en IA generativa es habilitar sistemas RAG (Retrieval-Augmented Generation), donde permite recuperar rápidamente los fragmentos de conocimiento más relevantes para que un modelo de lenguaje genere respuestas fundamentadas y actualizadas.


### El interruptor que hace lo contrario de lo que parece

`separar_razonamiento=False` **no apaga el razonamiento**. GLM razona siempre; lo único
que cambia es si vLLM devuelve esa cadena en su propio campo o la deja dentro de la
respuesta. Con `False`, lo que llega es el monólogo interno del modelo —en inglés— y
detrás, si alcanza el presupuesto, la respuesta.

Mirar las dos columnas de abajo: **los tokens de salida son casi los mismos**. Ahí está el
argumento: lo que se paga no cambia, porque el razonamiento ocurre igual.

In [4]:
if HAY_H200:
    for separar in (True, False):
        r = h.chat("¿Qué es un índice HNSW? Responde en dos frases.",
                   separar_razonamiento=separar, max_tokens=1500)
        print(f"\nseparar_razonamiento={str(separar):5s} → {r['tokens_salida']} tok de "
              f"salida · {r['latencia_s']:.1f} s · parada por {r['razon_de_parada']}")
        print(f"   campo razonamiento: {len(r['razonamiento'])} caracteres")
        print(f"   la respuesta empieza así: {r['texto'][:110]!r}")


separar_razonamiento=True  → 480 tok de salida · 3.7 s · parada por stop
   campo razonamiento: 1700 caracteres
   la respuesta empieza así: 'Un índice HNSW (Hierarchical Navigable Small World) es una estructura de datos basada en grafos que permite re'

separar_razonamiento=False → 592 tok de salida · 4.2 s · parada por stop
   campo razonamiento: 0 caracteres
   la respuesta empieza así: 'The user is asking in Spanish: "What is an HNSW index? Answer in two sentences."\n\nHNSW stands for Hierarchical'


## 3 · Los embeddings

Un embedding es un vector que representa el **significado** de un texto. Dos frases que
quieren decir lo mismo caen cerca aunque no compartan una sola palabra; dos frases que
comparten palabras pueden caer lejos. Eso es lo que una base de datos vectorial indexa.

In [5]:
if HAY_H200:
    FRASES = [
        "El gato duerme sobre el sofá.",
        "Un felino descansa en el mueble de la sala.",   # mismo significado, otras palabras
        "El gato de mi vecina se llama Pixel.",          # comparte palabras, otro asunto
        "Qdrant indexa vectores con HNSW.",              # otro tema por completo
    ]
    t0 = time.perf_counter()
    V = h.embeddings(FRASES)
    print(f"modelo    : {h.modelo_embeddings}")
    print(f"matriz    : {V.shape} · {V.dtype} · {time.perf_counter() - t0:.2f} s")
    print(f"normas    : {np.linalg.norm(V, axis=1).round(4)}  (a norma 1 → el producto "
          f"punto ES el coseno)")

modelo    : bge-m3:latest
matriz    : (4, 1024) · float32 · 0.29 s
normas    : [1. 1. 1. 1.]  (a norma 1 → el producto punto ES el coseno)


### La matriz de similitud, que es el argumento entero

Con los vectores normalizados, `V @ V.T` es la matriz de cosenos. Lo que hay que mirar no
son los números sino **dos comparaciones concretas**: la frase 0 contra la 1 (mismo
significado, ni una palabra de contenido en común) y la 0 contra la 2 (tres palabras en
común, asuntos distintos).

In [6]:
if HAY_H200:
    S = V @ V.T
    print("       " + "".join(f"{j:>8d}" for j in range(len(FRASES))))
    for i, fila in enumerate(S):
        print(f"  [{i}] " + "".join(f"{x:8.3f}" for x in fila) + f"   {FRASES[i][:46]}")
    print(f"\n  0↔1 (mismo significado, otras palabras): {S[0, 1]:.3f}")
    print(f"  0↔2 (palabras compartidas, otro asunto): {S[0, 2]:.3f}")
    print(f"  0↔3 (nada que ver)                     : {S[0, 3]:.3f}")
    print(f"\n  ¿el significado gana a las palabras compartidas? "
          f"{'SÍ' if S[0, 1] > S[0, 2] else 'NO'}")

              0       1       2       3
  [0]    1.000   0.653   0.509   0.270   El gato duerme sobre el sofá.
  [1]    0.653   1.000   0.546   0.309   Un felino descansa en el mueble de la sala.
  [2]    0.509   0.546   1.000   0.339   El gato de mi vecina se llama Pixel.
  [3]    0.270   0.309   0.339   1.000   Qdrant indexa vectores con HNSW.

  0↔1 (mismo significado, otras palabras): 0.653
  0↔2 (palabras compartidas, otro asunto): 0.509
  0↔3 (nada que ver)                     : 0.270

  ¿el significado gana a las palabras compartidas? SÍ


## 4 · Búsqueda semántica en diez líneas

Esto es, sin índice y sin base de datos, **lo que hace una base vectorial**: convertir la
consulta al mismo espacio y ordenar por similitud. Todo lo que añaden Qdrant, pgvector o
Weaviate —HNSW, IVF, filtros, persistencia— es hacer esto rápido sobre millones de
vectores sin compararlos todos. El resultado que buscan es el mismo que este.

In [7]:
CORPUS = [
    "HNSW construye un grafo navegable por capas y busca en tiempo logarítmico.",
    "IVF particiona el espacio en clústeres y solo revisa los nprobe más cercanos.",
    "El recall@k mide qué fracción de los k vecinos reales recupera el índice.",
    "Qdrant permite filtrar por metadata durante la búsqueda, no después.",
    "pgvector añade un tipo de dato vectorial a PostgreSQL y evita una base aparte.",
    "La cuantización de producto comprime los vectores y degrada algo el recall.",
    "Un chunk demasiado grande diluye el embedding y recupera pasajes irrelevantes.",
    "El coseno y el producto punto coinciden cuando los vectores están normalizados.",
]

if HAY_H200:
    E = h.embeddings(CORPUS)

    def buscar(consulta, k=3):
        """Los k fragmentos más parecidos a la consulta. Sin índice: se comparan todos."""
        q = h.embeddings([consulta])[0]
        puntajes = E @ q                      # coseno, porque todo está normalizado
        return [(float(puntajes[i]), CORPUS[i]) for i in np.argsort(-puntajes)[:k]]

    for consulta in ("¿cómo acelero la búsqueda sin revisar todos los vectores?",
                     "¿qué pasa si mis fragmentos son muy largos?"):
        print(f"\n▸ {consulta}")
        for puntaje, texto in buscar(consulta):
            print(f"   {puntaje:.3f}  {texto}")


▸ ¿cómo acelero la búsqueda sin revisar todos los vectores?
   0.573  Qdrant permite filtrar por metadata durante la búsqueda, no después.
   0.459  pgvector añade un tipo de dato vectorial a PostgreSQL y evita una base aparte.
   0.439  IVF particiona el espacio en clústeres y solo revisa los nprobe más cercanos.

▸ ¿qué pasa si mis fragmentos son muy largos?
   0.591  Un chunk demasiado grande diluye el embedding y recupera pasajes irrelevantes.
   0.448  Qdrant permite filtrar por metadata durante la búsqueda, no después.
   0.412  HNSW construye un grafo navegable por capas y busca en tiempo logarítmico.


### El primer resultado no siempre es el bueno, y conviene no taparlo

En la corrida del **2026-09-22**, la consulta «¿cómo acelero la búsqueda sin revisar todos
los vectores?» devolvió como primer resultado la línea de **Qdrant y los filtros por
metadata** (0,573), por delante de IVF (0,439), que es la respuesta correcta. La segunda
consulta sí acertó de pleno.

No es un fallo del código ni del modelo: es **cómo se comporta la similitud de embeddings**
cuando varios fragmentos comparten vocabulario de dominio. Y es exactamente la razón de que
existan las tres cosas que vienen después en el curso:

- **`recall@k` en vez de «acertó o no»** (sesión 09): con k=3 el fragmento correcto está
  ahí; medir solo el primer puesto habría dicho que el sistema falló.
- **El reranking con un cross-encoder** (sesión 08): reordenar esos 3 leyendo consulta y
  fragmento juntos, en vez de comparar dos vectores calculados por separado.
- **La fragmentación** (sesión 08): fragmentos que mezclan dos ideas producen vectores que
  no son de ninguna.

Si esta celda diera siempre el resultado perfecto, las tres sesiones siguientes no tendrían
problema que resolver.

## 5 · Las dos mitades juntas: un RAG mínimo

Recuperar con embeddings y **responder con el LLM a partir de lo recuperado**. Son quince
líneas y ya tiene la forma del Taller 2: lo que falta —fragmentación, una base vectorial
de verdad, métricas— es ingeniería sobre este esqueleto, no otro diseño.

La instrucción de abajo hace lo que casi nadie hace y todo el mundo debería: **le permite
al modelo decir que no sabe**. Un RAG que obliga a responder inventa cuando no recupera, y
la segunda pregunta está elegida para que no haya nada que recuperar.

In [8]:
if HAY_H200:
    def responder(pregunta, k=3):
        recuperados = buscar(pregunta, k)
        contexto = "\n".join(f"[{i}] {t}" for i, (_, t) in enumerate(recuperados))
        prompt = (f"Contexto:\n{contexto}\n\nPregunta: {pregunta}\n\n"
                  f"Responde SOLO con lo que diga el contexto y cita el número entre "
                  f"corchetes. Si el contexto no alcanza, dilo en una frase.")
        return h.chat(prompt, temperature=0.1, max_tokens=1200), recuperados

    for pregunta in ("¿Por qué conviene normalizar los vectores?",
                     "¿Cuánto cuesta una licencia de Qdrant Cloud?"):   # fuera del corpus
        r, recuperados = responder(pregunta)
        print(f"\n▸ {pregunta}")
        print(f"   recuperado: {', '.join(f'{p:.2f}' for p, _ in recuperados)}")
        print(f"   respuesta : {r['texto']}")
        print(f"   ({r['tokens_salida']} tok · {r['latencia_s']:.1f} s)")


▸ ¿Por qué conviene normalizar los vectores?
   recuperado: 0.62, 0.52, 0.43
   respuesta : Porque cuando los vectores están normalizados, el coseno y el producto punto coinciden [0].
   (378 tok · 2.9 s)

▸ ¿Cuánto cuesta una licencia de Qdrant Cloud?
   recuperado: 0.45, 0.36, 0.33
   respuesta : El contexto no contiene información sobre el precio de una licencia de Qdrant Cloud; solo menciona filtrado por metadata [0], tamaño de chunks [1] y cuantización de producto [2].
   (201 tok · 1.5 s)


## 6 · Cuando falla

| Síntoma | Qué es | Qué hacer |
|----------------------------------------------|-------------------------------------|---------------------------------------------------|
| `No hay respuesta de 172.28.230.10:12555`    | la VPN, casi siempre                | conectar GlobalProtect; `route -n get 172.28.230.10` debe salir por un `utun` |
| `404 · The model ... does not exist`         | un id escrito a mano que caducó     | no escribirlo: `h.modelo_chat` lo resuelve         |
| la respuesta empieza en inglés con «The user asks…» | `separar_razonamiento=False` | quitarlo: el default ya es el bueno                |
| `razon_de_parada` = `length`                 | el razonamiento se comió el cupo    | subir `max_tokens` (una respuesta corta pide ~600) |
| `no sirve ningún modelo de tipo embeddings`  | el puerto de Ollama no está en la lista | `H200(puertos=(12555, 12559, 11434))`          |
| una petición tarda un minuto la primera vez  | Ollama está cargando el modelo      | `h.cargados()` dice cuáles están calientes         |

## 🧑‍🏫 Cierre — cómo encaja con la sesión 07 y el Taller 2

- La Parte 4 es **la sesión 07 sin índice**: comparar la consulta contra todos los
  vectores. Cuando en clase aparezcan HNSW e IVF, la pregunta que los motiva ya está
  planteada aquí —¿y si en vez de 8 fragmentos son 8 millones?—.
- La Parte 5 es **el esqueleto del Taller 2**. Lo que el taller añade es fragmentación,
  persistencia y métricas; el flujo es este.
- **Los embeddings del taller son gratis y rápidos**: 39 ms por texto con `bge-m3`
  caliente. Un corpus de 5 000 fragmentos son unos tres minutos y cero dólares, así que
  nadie tiene excusa para entregar un corpus de juguete.
- **La dimensión es 1024** con `bge-m3`. Es el número que va en la configuración de la
  colección de Qdrant, y cambiar de modelo de embeddings obliga a reindexar: no son
  espacios compatibles.
- Lo que **no** cubre este cuaderno y el taller sí: qué pasa cuando el laboratorio está
  saturado. Conviene que cada grupo tenga un plan B —`sentence-transformers` en local, ya
  instalado— y que lo declare en el informe.